# 09 - Gradio Demo — Intel Image Classification

Notebook này dùng để demo các model chính của đồ án:

1. **ResNet18 Fine-tuned**  
   `models/cs231/resnet18_finetuned_best.pth`

2. **HOG + Logistic Regression tuned**  
   `models/cs114/best_tuned_model_full.joblib`

3. **Simple CNN**  
   `models/cs231/simple_cnn_best.pth`

4. **SVM + HOG + RGB Histogram**  
   Model `.pkl` bổ sung dùng pipeline:

```text
Ảnh đầu vào → Resize 128x128 → HOG orientations=12 → RGB Histogram bins=(8,8,8) → Concatenate → SVM
```

Bản notebook này giữ nguyên pipeline của **CNN**, **ResNet18**, và **HOG + Logistic Regression tuned**.
**SVM + HOG + RGB Histogram**.

Các số chiều quan trọng:

```text
Logistic Regression tuned : Resize 64x64  → HOG 1764 features
External SVM .pkl         : Resize 128x128 → HOG 10800 + RGB Hist 512 = 11312 features
```

In [1]:
!pip -q install gradio scikit-image joblib opencv-python-headless

In [2]:
import json
import joblib
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms

from skimage.feature import hog

import gradio as gr


## 1. Mount Google Drive và cấu hình đường dẫn

Notebook này dùng đường dẫn cố định theo project hiện tại của bạn. Nếu project nằm ở vị trí khác, chỉ cần sửa biến `PROJECT_ROOT`.


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# =========================================================
# PROJECT ROOT
# =========================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project"
)

# =========================================================
# MODEL PATHS
# =========================================================

RESNET18_PATH = PROJECT_ROOT / "models" / "cs231" / "resnet18_finetuned_best.pth"
LOGREG_PATH = PROJECT_ROOT / "models" / "cs114" / "best_tuned_model_full.joblib"
SIMPLE_CNN_PATH = PROJECT_ROOT / "models" / "cs231" / "simple_cnn_best.pth"



EXTERNAL_PKL_PATH = "/content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/CS231/best_hog_rgb_svm.pkl"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESNET18_PATH:", RESNET18_PATH)
print("LOGREG_PATH:", LOGREG_PATH)
print("SIMPLE_CNN_PATH:", SIMPLE_CNN_PATH)
print("EXTERNAL_PKL_PATH:", EXTERNAL_PKL_PATH)

assert PROJECT_ROOT.exists(), f"Không tìm thấy PROJECT_ROOT: {PROJECT_ROOT}"
assert RESNET18_PATH.exists(), f"Không tìm thấy model ResNet18: {RESNET18_PATH}"
assert LOGREG_PATH.exists(), f"Không tìm thấy model Logistic Regression: {LOGREG_PATH}"
assert SIMPLE_CNN_PATH.exists(), f"Không tìm thấy model Simple CNN: {SIMPLE_CNN_PATH}"

if EXTERNAL_PKL_PATH is not None:
    EXTERNAL_PKL_PATH = Path(EXTERNAL_PKL_PATH)
    assert EXTERNAL_PKL_PATH.exists(), f"Không tìm thấy model .pkl: {EXTERNAL_PKL_PATH}"

print("\nTất cả đường dẫn model chính đã sẵn sàng.")

PROJECT_ROOT: /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project
RESNET18_PATH: /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/models/cs231/resnet18_finetuned_best.pth
LOGREG_PATH: /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/models/cs114/best_tuned_model_full.joblib
SIMPLE_CNN_PATH: /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/intel_image_project/models/cs231/simple_cnn_best.pth
EXTERNAL_PKL_PATH: /content/drive/MyDrive/hk2 2025 - 2026/ CS231.Q22_Nhap_mon_thi_giac/CS231/best_hog_rgb_svm.pkl

Tất cả đường dẫn model chính đã sẵn sàng.


## 2. Cấu hình class và transform ảnh

Thứ tự class đang dùng:

```text
buildings, forest, glacier, mountain, sea, street
```


In [5]:
CLASS_NAMES = [
    "buildings",
    "forest",
    "glacier",
    "mountain",
    "sea",
    "street"
]

NUM_CLASSES = len(CLASS_NAMES)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", DEVICE)
print("CLASS_NAMES:", CLASS_NAMES)
print("NUM_CLASSES:", NUM_CLASSES)


# =========================================================
# Transform cho ResNet18 Fine-tuned
# =========================================================
# ResNet18 pretrained ImageNet dùng input 224x224 và ImageNet mean/std.

RESNET18_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# =========================================================
# Transform cho Simple CNN
# =========================================================
# Project của bạn dùng ảnh resize 150x150.
# Nếu Simple CNN lúc train dùng size khác, sửa SIMPLE_CNN_IMAGE_SIZE.

SIMPLE_CNN_IMAGE_SIZE = 150

DATASET_MEAN = [0.43018116, 0.45747542, 0.45382798]
DATASET_STD = [0.26941103, 0.26793626, 0.29834034]

SIMPLE_CNN_TRANSFORM = transforms.Compose([
    transforms.Resize((SIMPLE_CNN_IMAGE_SIZE, SIMPLE_CNN_IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=DATASET_MEAN,
        std=DATASET_STD
    )
])

DEVICE: cpu
CLASS_NAMES: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
NUM_CLASSES: 6


## 3. Hàm hỗ trợ load checkpoint PyTorch

Hàm này xử lý các kiểu lưu phổ biến:

```python
torch.save(model.state_dict(), path)
torch.save({"model_state_dict": model.state_dict()}, path)
torch.save({"state_dict": model.state_dict()}, path)
```


In [6]:
def clean_state_dict_keys(state_dict):
    new_state_dict = {}

    for key, value in state_dict.items():
        new_key = key

        if new_key.startswith("module."):
            new_key = new_key[len("module."):]

        if new_key.startswith("model."):
            new_key = new_key[len("model."):]

        new_state_dict[new_key] = value

    return new_state_dict


def extract_state_dict(checkpoint):
    if isinstance(checkpoint, nn.Module):
        return checkpoint

    if isinstance(checkpoint, dict):
        possible_keys = [
            "model_state_dict",
            "state_dict",
            "net_state_dict",
            "weights"
        ]

        for key in possible_keys:
            if key in checkpoint:
                value = checkpoint[key]

                if isinstance(value, dict):
                    return clean_state_dict_keys(value)

                if isinstance(value, nn.Module):
                    return value

        # Trường hợp checkpoint tự nó là state_dict
        if all(isinstance(k, str) for k in checkpoint.keys()):
            return clean_state_dict_keys(checkpoint)

    raise ValueError("Không nhận diện được định dạng checkpoint.")


def load_torch_checkpoint(path):
    checkpoint = torch.load(path, map_location=DEVICE)
    return extract_state_dict(checkpoint)


def class_id_to_name(class_id):
    if isinstance(class_id, str):
        return class_id

    class_id = int(class_id)

    if 0 <= class_id < len(CLASS_NAMES):
        return CLASS_NAMES[class_id]

    return str(class_id)

## 4. Load ResNet18 Fine-tuned

Đây là model demo chính nên dùng khi báo cáo.


In [7]:
def build_resnet18(num_classes):
    model = models.resnet18(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


resnet18_model = build_resnet18(NUM_CLASSES)

state_or_model = load_torch_checkpoint(RESNET18_PATH)

if isinstance(state_or_model, nn.Module):
    resnet18_model = state_or_model
else:
    resnet18_model.load_state_dict(state_or_model, strict=True)

resnet18_model = resnet18_model.to(DEVICE)
resnet18_model.eval()

print("Load ResNet18 Fine-tuned thành công.")

Load ResNet18 Fine-tuned thành công.


## 5. Định nghĩa và load Simple CNN

Class `SimpleCNN` dưới đây cần khớp với kiến trúc lúc train trong notebook `06_cs231_cnn_train.ipynb`.

Cell sẽ load trực tiếp được checkpoint `simple_cnn_best.pth`.


In [8]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 6, dropout: float = 0.4):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            # Giảm feature map về 1x1 để classifier gọn hơn.
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x


simple_cnn_model = SimpleCNN(num_classes=NUM_CLASSES, dropout=0.4)

state_or_model = load_torch_checkpoint(SIMPLE_CNN_PATH)

if isinstance(state_or_model, nn.Module):
    simple_cnn_model = state_or_model
else:
    simple_cnn_model.load_state_dict(state_or_model, strict=True)

simple_cnn_model = simple_cnn_model.to(DEVICE)
simple_cnn_model.eval()

print("Load Simple CNN thành công.")


Load Simple CNN thành công.


## 6. Load HOG + Logistic Regression và model `.pkl`

Notebook này có 2 pipeline sklearn khác nhau:

### Pipeline 1 — Logistic Regression tuned

Giữ nguyên giống lúc train trong `04_cs114_tuning.ipynb`:

```text
Ảnh đầu vào → Resize 64x64 → Grayscale → HOG 1764 features → StandardScaler → Logistic Regression
```

### Pipeline 2 — External `.pkl` model: SVM + HOG + RGB Histogram

Dùng đúng pipeline :

```text
Ảnh đầu vào → Resize 128x128 → HOG orientations=12 → RGB Histogram bins=(8,8,8) → Concatenate → SVM
```

Số chiều feature dự kiến của model `.pkl`:

```text
HOG 128x128 = 10800 features
RGB Histogram = 512 features
Tổng = 11312 features
```


In [9]:
logreg_model = joblib.load(LOGREG_PATH)

print("Load HOG + Logistic Regression tuned thành công.")
print("Kiểu object sau khi load:", type(logreg_model))
if isinstance(logreg_model, dict):
    print("Các key trong file joblib:", list(logreg_model.keys()))


external_pkl_model = None

if EXTERNAL_PKL_PATH is not None:
    external_pkl_model = joblib.load(EXTERNAL_PKL_PATH)
    print("Load external .pkl model thành công.")
    print("Kiểu object sau khi load:", type(external_pkl_model))
    if isinstance(external_pkl_model, dict):
        print("Các key trong file pkl:", list(external_pkl_model.keys()))
else:
    print("Chưa cấu hình EXTERNAL_PKL_PATH.")

Load HOG + Logistic Regression tuned thành công.
Kiểu object sau khi load: <class 'dict'>
Các key trong file joblib: ['model', 'experiment', 'model_type', 'feature_type', 'class_to_idx', 'idx_to_class', 'class_names', 'config', 'best_params', 'validation_metrics', 'test_metrics']
Load external .pkl model thành công.
Kiểu object sau khi load: <class 'sklearn.svm._classes.SVC'>


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## 7. Trích xuất feature cho các model sklearn

Phần này tách rõ 2 pipeline:

```text
Logistic Regression tuned:
Resize 64x64 → Grayscale → HOG 1764 features

External SVM .pkl:
Resize 128x128 → HOG 10800 features + RGB Histogram 512 features = 11312 features
```

Không dùng chung feature extractor cho 2 model này, vì chúng được train bằng cấu hình khác nhau.


In [19]:
# =========================================================
# FEATURE EXTRACTION 1:
# Logistic Regression tuned = HOG only, resize 64x64
# Giữ nguyên pipeline đúng với notebook 04_cs114_tuning.ipynb
# =========================================================

DEFAULT_LOGREG_HOG_IMAGE_SIZE = (64, 64)

LOGREG_HOG_CONFIG = {
    "orientations": 9,
    "pixels_per_cell": (8, 8),
    "cells_per_block": (2, 2),
    "block_norm": "L2-Hys",
    "feature_vector": True
}


def get_logreg_hog_image_size_from_model_bundle(model_bundle):
    """
    Ưu tiên lấy HOG size từ package đã lưu trong file joblib/pkl.
    Nếu không có config, dùng mặc định đúng với notebook 04_cs114_tuning: (64, 64).
    """

    if isinstance(model_bundle, dict):
        config = model_bundle.get("config", {})

        if isinstance(config, dict) and "hog_size" in config:
            hog_size = config["hog_size"]
            return tuple(int(x) for x in hog_size)

    return DEFAULT_LOGREG_HOG_IMAGE_SIZE


def pil_to_gray_like_logreg_training(pil_image, size):
    """
    Mô phỏng đúng preprocessing lúc train Logistic Regression:
    PIL RGB -> resize bằng PIL -> numpy float32 [0, 1] -> grayscale thủ công.
    """

    image = pil_image.convert("RGB")
    image = image.resize(size)

    arr = np.asarray(image, dtype=np.float32) / 255.0

    gray = (
        0.299 * arr[:, :, 0]
        + 0.587 * arr[:, :, 1]
        + 0.114 * arr[:, :, 2]
    )

    return gray


def extract_logreg_hog_feature_from_pil(pil_image, model_bundle=None):
    """
    Trích xuất HOG feature cho model HOG + Logistic Regression tuned.

    Output dự kiến:
    - shape = (1, 1764)
    """

    hog_image_size = get_logreg_hog_image_size_from_model_bundle(model_bundle)

    gray = pil_to_gray_like_logreg_training(
        pil_image,
        hog_image_size
    )

    feature = hog(
        gray,
        orientations=LOGREG_HOG_CONFIG["orientations"],
        pixels_per_cell=LOGREG_HOG_CONFIG["pixels_per_cell"],
        cells_per_block=LOGREG_HOG_CONFIG["cells_per_block"],
        block_norm=LOGREG_HOG_CONFIG["block_norm"],
        feature_vector=LOGREG_HOG_CONFIG["feature_vector"]
    )

    return feature.astype(np.float32).reshape(1, -1)


# Giữ alias cũ để tránh lỗi nếu các cell phía sau vẫn gọi tên cũ.
extract_hog_feature_from_pil = extract_logreg_hog_feature_from_pil
get_hog_image_size_from_model_bundle = get_logreg_hog_image_size_from_model_bundle


# =========================================================
# FEATURE EXTRACTION 2:
# SVM + HOG + RGB Histogram, resize 128x128
# =========================================================

EXTERNAL_SVM_IMAGE_SIZE = (128, 128)

EXTERNAL_HOG_ORIENTATIONS = 12
EXTERNAL_HOG_PIXELS_PER_CELL = (8, 8)
EXTERNAL_HOG_CELLS_PER_BLOCK = (2, 2)
EXTERNAL_HOG_BLOCK_NORM = "L2-Hys"

EXTERNAL_RGB_HIST_BINS = (8, 8, 8)


def pil_to_external_rgb_uint8_128(pil_image):
    """
    Chuyển ảnh PIL từ Gradio về đúng format cho model SVM .pkl:
    - RGB
    - resize 128x128
    - uint8, pixel nằm trong [0, 255]

    Lưu ý: Không chia ảnh cho 255 vì cv2.calcHist đang dùng range [0, 256].
    """

    image = pil_image.convert("RGB")
    image = np.asarray(image, dtype=np.uint8)

    image = cv2.resize(
        image,
        EXTERNAL_SVM_IMAGE_SIZE,
        interpolation=cv2.INTER_AREA
    )

    if image.dtype != np.uint8:
        image = image.astype(np.uint8)

    return image


def extract_external_hog_features(
    images,
    orientations=EXTERNAL_HOG_ORIENTATIONS,
    pixels_per_cell=EXTERNAL_HOG_PIXELS_PER_CELL,
    cells_per_block=EXTERNAL_HOG_CELLS_PER_BLOCK
):
    """
    Hàm này giữ đúng logic bạn đã dùng lúc train SVM:
    RGB image -> cv2.COLOR_RGB2GRAY -> skimage.feature.hog.
    """

    features = []

    for img in images:
        gray = cv2.cvtColor(
            img,
            cv2.COLOR_RGB2GRAY
        )

        hog_feature = hog(
            gray,
            orientations=orientations,
            pixels_per_cell=pixels_per_cell,
            cells_per_block=cells_per_block,
            block_norm=EXTERNAL_HOG_BLOCK_NORM,
            feature_vector=True
        )

        features.append(hog_feature)

    return np.array(features)


def extract_external_rgb_histogram(
    images,
    bins=EXTERNAL_RGB_HIST_BINS
):
    """
    Hàm này giữ đúng logic bạn đã dùng lúc train SVM:
    cv2.calcHist trên 3 kênh của ảnh RGB uint8, sau đó normalize và flatten.
    """

    features = []

    for img in images:
        hist = cv2.calcHist(
            [img],
            [0, 1, 2],
            None,
            bins,
            [0, 256, 0, 256, 0, 256]
        )

        hist = cv2.normalize(
            hist,
            hist
        ).flatten()

        features.append(hist)

    return np.array(features)


def build_external_svm_hog_rgb_features(images):
    """
    Feature cuối cho External PKL SVM:
    HOG 10800 + RGB Histogram 512 = 11312 features.
    """

    hog_features = extract_external_hog_features(
        images=images,
        orientations=EXTERNAL_HOG_ORIENTATIONS,
        pixels_per_cell=EXTERNAL_HOG_PIXELS_PER_CELL,
        cells_per_block=EXTERNAL_HOG_CELLS_PER_BLOCK
    )

    rgb_features = extract_external_rgb_histogram(
        images=images,
        bins=EXTERNAL_RGB_HIST_BINS
    )

    final_features = np.concatenate(
        [hog_features, rgb_features],
        axis=1
    )

    return final_features


def extract_external_svm_hog_rgb_feature_from_pil(pil_image):
    """
    Hàm dùng riêng cho model .pkl SVM + HOG + RGB.

    Input:
    - 1 ảnh PIL từ Gradio

    Output:
    - feature shape = (1, 11312)
    """

    image = pil_to_external_rgb_uint8_128(pil_image)

    features = build_external_svm_hog_rgb_features(
        [image]
    )

    return features.astype(np.float32)


# =========================================================
# KIỂM TRA NHANH FEATURE DIMENSION
# =========================================================

LOGREG_HOG_IMAGE_SIZE = get_logreg_hog_image_size_from_model_bundle(logreg_model)

_dummy_img = Image.new("RGB", (150, 150))

_logreg_dummy_feature = extract_logreg_hog_feature_from_pil(
    _dummy_img,
    model_bundle=logreg_model
)

_external_dummy_feature = extract_external_svm_hog_rgb_feature_from_pil(
    _dummy_img
)

print("LOGREG_HOG_IMAGE_SIZE:", LOGREG_HOG_IMAGE_SIZE)
print("Logistic Regression HOG feature shape:", _logreg_dummy_feature.shape)

if _logreg_dummy_feature.shape[1] != 1764:
    print(
        "Cảnh báo: HOG feature của Logistic Regression không phải 1764 chiều. "
        "Hãy kiểm tra lại config HOG trong model joblib/pkl."
    )
else:
    print("OK Logistic Regression: HOG feature = 1764 features.")

print("\nEXTERNAL_SVM_IMAGE_SIZE:", EXTERNAL_SVM_IMAGE_SIZE)
print("SVM HOG + RGB feature shape:", _external_dummy_feature.shape)

if _external_dummy_feature.shape[1] != 11312:
    print(
        "Cảnh báo: feature của External SVM không phải 11312 chiều. "
        "Hãy kiểm tra lại resize, HOG config hoặc RGB histogram bins."
    )
else:
    print("OK External SVM: HOG + RGB feature = 11312 features.")

LOGREG_HOG_IMAGE_SIZE: (64, 64)
Logistic Regression HOG feature shape: (1, 1764)
OK Logistic Regression: HOG feature = 1764 features.

EXTERNAL_SVM_IMAGE_SIZE: (128, 128)
SVM HOG + RGB feature shape: (1, 11312)
OK External SVM: HOG + RGB feature = 11312 features.


## 8. Hàm dự đoán

Có 2 nhóm model:

```text
PyTorch model:
- ResNet18 Fine-tuned
- Simple CNN

sklearn/joblib/pkl model:
- HOG + Logistic Regression tuned: dùng HOG 64x64, 1764 features
- SVM: dùng HOG + RGB Histogram 128x128, 11312 features
```


In [20]:
def torch_predict(model, pil_image, transform):
    image = pil_image.convert("RGB")
    x = transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)[0].detach().cpu().numpy()

    top_indices = np.argsort(probs)[::-1]

    result = {}

    for idx in top_indices:
        class_name = class_id_to_name(idx)
        result[class_name] = float(probs[idx])

    pred_idx = int(top_indices[0])
    pred_label = class_id_to_name(pred_idx)
    confidence = float(probs[pred_idx])

    return pred_label, confidence, result


def get_sklearn_estimator(model_bundle):
    """
    File .joblib/.pkl có thể lưu trực tiếp model sklearn,
    hoặc lưu dạng dict chứa model + metadata/config/scaler.
    Hàm này lấy đúng object có hàm predict().
    """

    if model_bundle is None:
        raise ValueError("Model .pkl/.joblib chưa được cấu hình.")

    if hasattr(model_bundle, "predict"):
        return model_bundle

    if isinstance(model_bundle, dict):
        preferred_keys = [
            "pipeline",
            "best_estimator",
            "best_estimator_",
            "model",
            "best_model",
            "estimator",
            "classifier",
            "clf",
            "logreg_model",
            "svm_model",
            "svm",
            "svc"
        ]

        for key in preferred_keys:
            if key in model_bundle and hasattr(model_bundle[key], "predict"):
                return model_bundle[key]

        # Nếu không biết key nào, quét toàn bộ dict để tìm object có predict().
        for key, value in model_bundle.items():
            if hasattr(value, "predict"):
                print(f"Tự động lấy estimator từ key: {key}")
                return value

        raise TypeError(
            "File model đang là dict nhưng không tìm thấy object nào có hàm predict(). "
            f"Các key hiện có: {list(model_bundle.keys())}"
        )

    raise TypeError(
        "Object sau khi load không phải sklearn model và cũng không phải dict chứa model. "
        f"Kiểu object hiện tại: {type(model_bundle)}"
    )


def apply_optional_preprocessors(model_bundle, feature, estimator):
    """
    Nếu file joblib/pkl lưu riêng scaler/PCA/selector ngoài model,
    áp dụng các bước transform này trước khi predict.

    Nếu estimator là sklearn Pipeline thì không cần làm gì thêm,
    vì Pipeline đã tự xử lý transform bên trong.
    """

    if not isinstance(model_bundle, dict):
        return feature

    if hasattr(estimator, "steps"):
        return feature

    preprocessor_keys = [
        "scaler",
        "standard_scaler",
        "feature_scaler",
        "normalizer",
        "pca",
        "selector",
        "feature_selector"
    ]

    x = feature

    for key in preprocessor_keys:
        if key in model_bundle and hasattr(model_bundle[key], "transform"):
            x = model_bundle[key].transform(x)

    return x


def get_expected_raw_feature_count(estimator):
    """
    Lấy số chiều feature mà model/scaler/pipeline đang mong đợi.
    Hữu ích để bắt lỗi mismatch trước khi predict.
    """

    if hasattr(estimator, "n_features_in_"):
        return int(estimator.n_features_in_)

    if hasattr(estimator, "named_steps"):
        for step_name, step_obj in estimator.named_steps.items():
            if hasattr(step_obj, "n_features_in_"):
                return int(step_obj.n_features_in_)

    if hasattr(estimator, "steps"):
        for step_name, step_obj in estimator.steps:
            if hasattr(step_obj, "n_features_in_"):
                return int(step_obj.n_features_in_)

    return None


def validate_raw_feature_dimension_before_transform(
    estimator,
    feature,
    model_name,
    pipeline_hint
):
    """
    Kiểm tra feature trước khi đi qua StandardScaler/Pipeline.
    Nếu số chiều sai, báo lỗi rõ ràng thay vì để sklearn báo lỗi khó hiểu.
    """

    expected_features = get_expected_raw_feature_count(estimator)

    if expected_features is None:
        return

    actual_features = int(feature.shape[1])

    if actual_features != expected_features:
        raise ValueError(
            f"Sai số chiều feature cho {model_name}. "
            f"Model đang cần {expected_features} features, nhưng pipeline hiện tạo ra {actual_features} features. "
            f"Pipeline đúng nên là: {pipeline_hint}"
        )


def print_sklearn_model_dimension_check(
    model_bundle,
    model_name,
    feature_extractor,
    pipeline_hint
):
    """
    In thông tin kiểm tra nhanh sau khi load model.
    """

    estimator = get_sklearn_estimator(model_bundle)
    expected_features = get_expected_raw_feature_count(estimator)

    dummy_img = Image.new("RGB", (150, 150))
    dummy_feature = feature_extractor(dummy_img)

    actual_features = int(dummy_feature.shape[1])

    print(f"[{model_name}] Current feature shape:", dummy_feature.shape)

    if expected_features is not None:
        print(f"[{model_name}] Model expected raw features:", expected_features)

        if expected_features == actual_features:
            print(f"[{model_name}] OK: feature dimension khớp.")
        else:
            print(
                f"[{model_name}] Cảnh báo: model cần {expected_features} features, "
                f"nhưng pipeline hiện tạo ra {actual_features} features."
            )
            print(f"[{model_name}] Pipeline đúng nên là: {pipeline_hint}")
    else:
        print(f"[{model_name}] Không đọc được expected feature từ model.")


def sklearn_predict_with_feature_extractor(
    model_bundle,
    pil_image,
    feature_extractor,
    model_name,
    pipeline_hint
):
    estimator = get_sklearn_estimator(model_bundle)

    feature = feature_extractor(pil_image)

    validate_raw_feature_dimension_before_transform(
        estimator=estimator,
        feature=feature,
        model_name=model_name,
        pipeline_hint=pipeline_hint
    )

    feature = apply_optional_preprocessors(
        model_bundle,
        feature,
        estimator
    )

    if hasattr(estimator, "predict_proba"):
        probs = estimator.predict_proba(feature)[0]

        if hasattr(estimator, "classes_"):
            classes = list(estimator.classes_)
        else:
            classes = list(range(len(probs)))

        top_indices = np.argsort(probs)[::-1]

        result = {}

        for idx in top_indices:
            class_name = class_id_to_name(classes[idx])
            result[class_name] = float(probs[idx])

        pred_label = class_id_to_name(classes[top_indices[0]])
        confidence = float(probs[top_indices[0]])

        return pred_label, confidence, result

    if hasattr(estimator, "decision_function"):
        scores = estimator.decision_function(feature)

        if scores.ndim == 2:
            scores = scores[0]

        exp_scores = np.exp(scores - np.max(scores))
        probs = exp_scores / exp_scores.sum()

        if hasattr(estimator, "classes_"):
            classes = list(estimator.classes_)
        else:
            classes = list(range(len(probs)))

        top_indices = np.argsort(probs)[::-1]

        result = {}

        for idx in top_indices:
            class_name = class_id_to_name(classes[idx])
            result[class_name] = float(probs[idx])

        pred_label = class_id_to_name(classes[top_indices[0]])
        confidence = float(probs[top_indices[0]])

        return pred_label, confidence, result

    pred = estimator.predict(feature)[0]
    pred_label = class_id_to_name(pred)

    result = {
        pred_label: 1.0
    }

    return pred_label, 1.0, result


def sklearn_predict_logreg_hog(model_bundle, pil_image):
    return sklearn_predict_with_feature_extractor(
        model_bundle=model_bundle,
        pil_image=pil_image,
        feature_extractor=lambda img: extract_logreg_hog_feature_from_pil(
            img,
            model_bundle=model_bundle
        ),
        model_name="HOG + Logistic Regression tuned",
        pipeline_hint="Resize 64x64 → Grayscale → HOG 1764 features → StandardScaler → Logistic Regression"
    )


def sklearn_predict_external_svm_hog_rgb(model_bundle, pil_image):
    return sklearn_predict_with_feature_extractor(
        model_bundle=model_bundle,
        pil_image=pil_image,
        feature_extractor=extract_external_svm_hog_rgb_feature_from_pil,
        model_name="SVM + HOG + RGB Histogram",
        pipeline_hint="Resize 128x128 → HOG orientations=12 → RGB Histogram bins=(8,8,8) → Concatenate 11312 features → SVM"
    )


# Kiểm tra số chiều feature sau khi đã có hàm get_sklearn_estimator().
print_sklearn_model_dimension_check(
    model_bundle=logreg_model,
    model_name="HOG + Logistic Regression tuned",
    feature_extractor=lambda img: extract_logreg_hog_feature_from_pil(
        img,
        model_bundle=logreg_model
    ),
    pipeline_hint="Resize 64x64 → Grayscale → HOG 1764 features → StandardScaler → Logistic Regression"
)

if external_pkl_model is not None:
    print_sklearn_model_dimension_check(
        model_bundle=external_pkl_model,
        model_name="SVM + HOG + RGB Histogram",
        feature_extractor=extract_external_svm_hog_rgb_feature_from_pil,
        pipeline_hint="Resize 128x128 → HOG orientations=12 → RGB Histogram bins=(8,8,8) → Concatenate 11312 features → SVM"
    )


[HOG + Logistic Regression tuned] Current feature shape: (1, 1764)
[HOG + Logistic Regression tuned] Model expected raw features: 1764
[HOG + Logistic Regression tuned] OK: feature dimension khớp.
[SVM + HOG + RGB Histogram] Current feature shape: (1, 11312)
[SVM + HOG + RGB Histogram] Model expected raw features: 11312
[SVM + HOG + RGB Histogram] OK: feature dimension khớp.


## 9. Hàm xử lý cho Gradio

In [21]:
MODEL_OPTIONS = [
    "ResNet18 Fine-tuned",
    "HOG + Logistic Regression tuned",
    "Simple CNN",
    "SVM + HOG + RGB"
]


def predict_with_selected_model(input_image, selected_model):
    if input_image is None:
        return {}, "Vui lòng upload một ảnh trước."

    pil_image = input_image.convert("RGB")

    try:
        if selected_model == "ResNet18 Fine-tuned":
            pred_label, confidence, result = torch_predict(
                resnet18_model,
                pil_image,
                RESNET18_TRANSFORM
            )

            note = (
                f"Model: ResNet18 Fine-tuned\n"
                f"File: {RESNET18_PATH}\n"
                f"Kết quả dự đoán: {pred_label}\n"
                f"Độ tin cậy: {confidence:.4f}\n\n"
                f"Pipeline: Ảnh đầu vào → Resize 224x224 → Normalize ImageNet → ResNet18 Fine-tuned."
            )

            return result, note

        if selected_model == "HOG + Logistic Regression tuned":
            pred_label, confidence, result = sklearn_predict_logreg_hog(
                logreg_model,
                pil_image
            )

            note = (
                f"Model: HOG + Logistic Regression tuned\n"
                f"File: {LOGREG_PATH}\n"
                f"Kết quả dự đoán: {pred_label}\n"
                f"Độ tin cậy/xác suất: {confidence:.4f}\n\n"
                f"Pipeline: Ảnh đầu vào → Resize {LOGREG_HOG_IMAGE_SIZE[0]}x{LOGREG_HOG_IMAGE_SIZE[1]} "
                f"→ Grayscale → HOG 1764 features → StandardScaler → Logistic Regression."
            )

            return result, note

        if selected_model == "Simple CNN":
            pred_label, confidence, result = torch_predict(
                simple_cnn_model,
                pil_image,
                SIMPLE_CNN_TRANSFORM
            )

            note = (
                f"Model: Simple CNN\n"
                f"File: {SIMPLE_CNN_PATH}\n"
                f"Kết quả dự đoán: {pred_label}\n"
                f"Độ tin cậy: {confidence:.4f}\n\n"
                f"Pipeline: Ảnh đầu vào → Resize  → Normalize dataset mean/std → Simple CNN."
            )

            return result, note

        if selected_model == "SVM + HOG + RGB":
            pred_label, confidence, result = sklearn_predict_external_svm_hog_rgb(
                external_pkl_model,
                pil_image
            )

            note = (
                f"Model: SVM + HOG + RGB Histogram\n"
                f"File: {EXTERNAL_PKL_PATH}\n"
                f"Kết quả dự đoán: {pred_label}\n"
                f"Độ tin cậy/xác suất quy đổi: {confidence:.4f}\n\n"
                f"Pipeline: Ảnh đầu vào → Resize 128x128 → HOG orientations=12 "
                f"+ RGB Histogram bins=(8,8,8) → Concatenate 11312 features → SVM."
            )

            return result, note

        return {}, "Model không hợp lệ."

    except Exception as e:
        return {}, f"Lỗi khi dự đoán với model {selected_model}:\n{e}"


## 10. Chạy giao diện Gradio

Sau khi chạy cell dưới, Colab sẽ tạo một đường link public để demo.


In [ ]:
description_text = f"""
# Intel Image Classification Demo

Upload một ảnh cảnh tự nhiên và chọn model để dự đoán một trong 6 lớp:

**{", ".join(CLASS_NAMES)}**

## Model demo

1. ResNet18 Fine-tuned — model có kết quả tốt nhất.
2. HOG + Logistic Regression tuned — model truyền thống.
   - Pipeline đúng: Resize 64x64 → Grayscale → HOG 1764 features → StandardScaler → Logistic Regression.
3. Simple CNN — baseline deep learning.
4. SVM + HOG + RGB.
   - Pipeline đúng: Resize 128x128 → HOG orientations=12 → RGB Histogram bins=(8,8,8) → Concatenate 11312 features → SVM.
"""


demo = gr.Interface(
    fn=predict_with_selected_model,
    inputs=[
        gr.Image(
            type="pil",
            label="Upload ảnh cần phân loại"
        ),
        gr.Dropdown(
            choices=MODEL_OPTIONS,
            value="ResNet18 Fine-tuned",
            label="Chọn model"
        )
    ],
    outputs=[
        gr.Label(
            num_top_classes=6,
            label="Xác suất dự đoán"
        ),
        gr.Textbox(
            label="Thông tin kết quả",
            lines=10
        )
    ],
    title="Intel Image Classification Demo",
    description=description_text
)

demo.launch(
    share=True,
    debug=True
)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://725620a9f9d7015d3b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Created dataset file at: .gradio/flagged/dataset1.csv
